In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np
import pickle


2026-09-21 20:25:01.811648: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
## Load trained model , scaler pickle , onehot
model = load_model('churn_model.keras')

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)




In [3]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}   

In [4]:
input_df = pd.DataFrame([input_data])

In [5]:
geo_encoded = onehot_encoder_geo.transform(
    pd.DataFrame(
        [[input_data['Geography']]],
        columns=['Geography']
    )
).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [6]:
## Enode categorical features
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
## concat one hot encoded geography columns with input_df
input_df = pd.concat([input_df.drop('Geography',axis=1), geo_encoded_df], axis=1)

In [8]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [9]:
## scaling input data
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.52378626,  0.90036493,  0.09471847, -0.69529215, -0.26157498,
         0.79949262,  0.6430943 ,  0.96655883, -0.86107284,  1.00314781,
        -0.57823004, -0.57888987]])

In [10]:
prediction = model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 109ms/step


array([[0.0462964]], dtype=float32)

In [11]:
prediction_proba = prediction[0][0]

In [12]:
probability = prediction_proba * 100
probability

4.629639536142349

In [13]:
if prediction_proba > 0.5:
    print(f"The customer is likely to churn ")
else:
    print(f"The customer is unlikely to churn ")     

The customer is unlikely to churn 
